# University of London - ML Code - Computer Science Final Project

**BSc Computer Science**

**Subject: CM3070 Computer Science Final Project**

**Student: In Final Project report**

**Student Number: In Final Project report**

## Experiment 19

Summary: This experiment seeks to test KerasTuner, a "general-purpose hyperparameter tuning library" [3], in order to test obtaining the best model before using it in the Genetic Algorithm.

## Preparing the data

### Transforming the csv data to a numpy array

In [1]:
import numpy as np

str_to_np_date = lambda x: np.datetime64(x)

usdYen_raw_data = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=1)
usdYen_raw_data_dates = np.genfromtxt("./data/currency-data/USD-JPY-DAILY.csv", skip_header=1, delimiter=";", usecols=0, converters={0: str_to_np_date})

print("Length: ",len(usdYen_raw_data))
print("Data type: ",usdYen_raw_data.dtype)
print("Raw Data: ",usdYen_raw_data)
print("Raw Data Dates: ",usdYen_raw_data_dates)

Length:  5000
Data type:  float64
Raw Data:  [154.71 155.21 155.81 ... 118.22 118.89 118.46]
Raw Data Dates:  ['2025-12-16' '2025-12-15' '2025-12-12' ... '2006-10-19' '2006-10-18'
 '2006-10-17']


As the currency data is from newer to older, the order should be inverted.

In [2]:
usdYen_raw_data = np.flip(usdYen_raw_data, axis=0)
print(usdYen_raw_data)

[118.46 118.89 118.22 ... 155.81 155.21 154.71]


### Computing the numer of samples for each data split

In [3]:
train_samples_number = len(usdYen_raw_data)
print("Number of train samples: ", train_samples_number)

Number of train samples:  5000


### Creating timeseries data

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [4]:
from tensorflow import keras

# Parameters
sampling_rate = 1
sequence_length = 150 # Observations will go back 150 days
delay = sampling_rate * (sequence_length + 5 - 1) # target is 5 days after the end of the sequence
batch_size = train_samples_number

# train dataset
train_dataset = keras.utils.timeseries_dataset_from_array(
    usdYen_raw_data,
    targets=usdYen_raw_data[delay:],
    sampling_rate=sampling_rate,
    sequence_length=sequence_length,
    batch_size=batch_size,
)

### - Checking that timeseries data works correctly

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [5]:
'''
for inputs, targets in train_dataset:
    for i in range(inputs.shape[0]):
        print([float(x) for x in inputs[i]], float(targets[i]))
'''

'\nfor inputs, targets in train_dataset:\n    for i in range(inputs.shape[0]):\n        print([float(x) for x in inputs[i]], float(targets[i]))\n'

### - Extracting data inputs and outputs

Based on the code provided in Chapter 10 (Deep Learning For Timeseries) of the book Deep Learning with Python by Francois Chollet [1].

In [6]:
import tensorflow as tf

data_inputs = []
data_outputs = []

for samples, targets in train_dataset:
    print("Samples: ", samples)
    print("Sample shape: ", samples.shape)
    print("Targets: ", targets)
    print("Targets shape: ", targets.shape)
    data_inputs = tf.make_ndarray(tf.make_tensor_proto(samples))
    data_outputs = tf.reshape(tf.make_ndarray(tf.make_tensor_proto(targets)), [-1,1])

data_inputs_test = data_inputs[-200:]
data_outputs_test = data_outputs[-200:]
data_inputs = data_inputs[:-200]
data_outputs = data_outputs[:-200]

print("Data Inputs: ", len(data_inputs))
print("Data Inputs Test: ", len(data_inputs_test))
print("Data Outputs: ", len(data_outputs))
print("Data Outputs Test: ", len(data_outputs_test))
    

Samples:  tf.Tensor(
[[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [148.35  147.453 146.63  ... 155.24  155.08  155.34 ]
 [147.453 146.63  145.592 ... 155.08  155.34  155.92 ]
 [146.63  145.592 145.581 ... 155.34  155.92  156.86 ]], shape=(4846, 150), dtype=float64)
Sample shape:  (4846, 150)
Targets:  tf.Tensor([121.45 121.54 121.65 ... 155.81 155.21 154.71], shape=(4846,), dtype=float64)
Targets shape:  (4846,)
Data Inputs:  4646
Data Inputs Test:  200
Data Outputs:  4646
Data Outputs Test:  200


2026-01-20 14:30:04.309318: I tensorflow/core/framework/local_rendezvous.cc:407] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence


In [7]:
print("----")
print("Input Data: ", data_inputs)
print("----")
print("Output Data: ", data_outputs)
print("----")

----
Input Data:  [[118.46  118.89  118.22  ... 119.88  120.18  120.34 ]
 [118.89  118.22  118.7   ... 120.18  120.34  120.26 ]
 [118.22  118.7   119.34  ... 120.34  120.26  120.8  ]
 ...
 [144.052 144.518 146.702 ... 149.076 149.801 150.545]
 [144.518 146.702 147.181 ... 149.801 150.545 149.485]
 [146.702 147.181 146.634 ... 150.545 149.485 149.801]]
----
Output Data:  tf.Tensor(
[[121.45 ]
 [121.54 ]
 [121.65 ]
 ...
 [148.018]
 [147.151]
 [147.708]], shape=(4646, 1), dtype=float64)
----


## HyperTuner

The code is based on the code provided in the Keras documentation for the KerasTuner [3].

In [8]:
from keras import models
from keras import layers

def build_lstm_model(hp):
    model = models.Sequential()
    model.add(
        layers.LSTM(
            units = hp.Int("units", min_value=16, max_value=64, step=4),
            input_shape=(sequence_length, 1)
        )
    )
    
    for i in range(2):
        model.add(
            layers.Dense(
                units = hp.Int(f"units_{i}", min_value=4, max_value=56, step=4)
            )
        )
    
    model.add(
        layers.Dense(
            units = 1
        )
    )
    model.compile(optimizer="rmsprop", loss="mse", metrics=["mae"])
    return model

In [9]:
import keras_tuner

tuner = keras_tuner.RandomSearch(
    hypermodel=build_lstm_model,
    objective="val_mae",
    max_trials=10,
    executions_per_trial=4,
    overwrite=True,
    directory="tuner_results/final_project_tuner_experimentNineteen_results",
    project_name="final_project_tunerfinal_project_tuner_experimentNineteen"
)

tuner.search_space_summary()

Search space summary
Default search space size: 3
units (Int)
{'default': None, 'conditions': [], 'min_value': 16, 'max_value': 64, 'step': 4, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 4, 'max_value': 56, 'step': 4, 'sampling': 'linear'}


/Users/studentcode/Documents/UOL/Semester 6/Computer Science Final Project/ML - CM3070 - In Final Project report - University of London/.venv/lib/python3.13/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [10]:
tuner.search(data_inputs, data_outputs, epochs=5, validation_data=(data_inputs_test, data_outputs_test))

Trial 10 Complete [00h 00m 59s]
val_mae: 2.392872631549835

Best val_mae So Far: 1.917658418416977
Total elapsed time: 00h 07m 32s


In [12]:
best_model = tuner.get_best_models(num_models=1)[0]
best_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 40)             │         6,720 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         1,312 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 20)             │           660 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            21 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,713 (34.04 KB)

 Trainable params: 8,713 (34.04 KB)

 Non-trainable params: 0 (0.00 B)

## Genetic Algorithm

The code for the Genetic Algorithm is based on the code provided by the PyGAD library documentation [2]. But for this experiment this is not necessary.

## About the code

The Genetic Algorithm code is based on the code shown in the docs of the PyGAD library.

The timeseries data code is based on the code shown in the chapter of the book Deep Learning with Python.

## References

1- Francois Chollet. 2021. Deep Learning with Python, Second Edition. Chapter 10, Deep learning for timeseries. Manning Publications.

2- PyGAD. pygad.kerasga Module. Retrieved from https://pygad.readthedocs.io/en/latest/kerasga.html#

3- Keras. Getting started with KerasTuner. Retrieved from https://keras.io/keras_tuner/getting_started/